In [2]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** XX  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [64]:
tickers = pd.read_csv('Tickers_Example.csv', header=None, names = ['Ticker'])

#created to filter all tickers based on assignment requirements. Returns a list of tickers that meet the requirements.
def filter_ticks (tickers, min_volume = 5000, min_days_per_month = 18):
    start_date = '2025-10-01'
    end_date = '2026-10-01'
    filtered = []

    for i in tickers['Ticker'].drop_duplicates(): #passes ticker column of the csv, dropping duplicates to avoid them appearing twice
        try:
            ticker = yf.Ticker(i)
            
            history = ticker.history(start=start_date, end=end_date)
            
            if (history.empty): #ticker~ is invalid or has no data for the specified date range
                continue

            vol = history['Volume']
            month = history.index.strftime('%Y-%m')  #masks to group by month
            avg_vol = vol[vol.groupby(month).transform('count') >= min_days_per_month] #filters data by making sure the number of days is greater than 18

            if ((avg_vol.mean() >= min_volume) or avg_vol.empty): #verifies stock has a daily volume of at least 5000 shares over the past year
                info = ticker.info #saves runtime by running this after the initial check
                market = info.get('market')
                if (market == 'us_market' or market == 'ca_market') and (info.get('currency') == 'USD' or info.get('currency') == 'CAD') and (info.get('quoteType') == 'EQUITY'): #verifies stock is listed on a US or Canadian exchange and is traded in USD or CAD
                    filtered.append(i)
        except:
            continue #in case ticker is invalid, ignore the current ticker

    return filtered
                    

filtered_tickers = filter_ticks(tickers)


def get_usd_cad_rate(): #converts price from USD to CAD
    start_date = '2026-11-20'
    end_date = '2026-11-21'
    return yf.Ticker('USDCAD=X').history(start=start_date, end=end_date)['Close'].iloc[0]

def get_close_prices(tickers, start_date = '2026-11-20', end_date = '2026-11-21'): #change start date to 2026-11-27 and end to 2026-11-28
    close_prices = pd.DataFrame()
    rate = get_usd_cad_rate()
    for ticker in filtered_tickers:
        try:
            data = yf.download(ticker, start=start_date, end=end_date, progress=False) #get data for all valid tickers
            if yf.Ticker(ticker).info.get('currency') == 'USD': #if price is listed in USD
                close_prices[ticker] = rate * data['Close']
            else:
                close_prices[ticker] = data['Close']
        except:
            continue

    return close_prices

all_close_prices = get_close_prices(filtered_tickers)
print(all_close_prices)

$AGN: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$CELG: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found


                  AAPL        ABBV         ABT         ACN         AIG  \
Date                                                                     
2026-11-19  500.128085  406.077504  148.759505  313.043729  113.857518   

                  AMZN         AXP          BA        BAC  BB.TO  ...  \
Date                                                              ...   
2026-11-19  368.597595  445.562398  229.877604  69.560498  11.94  ...   

                 PYPL        QCOM     SHOP.TO  T.TO       TD.TO         TXN  \
Date                                                                          
2026-11-19  80.731423  232.025859  189.350006  15.5  165.270004  409.242587   

                   UNH         UNP         UPS        USB  
Date                                                       
2026-11-19  473.303568  416.231584  156.264073  82.006054  

[1 rows x 35 columns]


## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.